# 02 — Feature Extraction

**Purpose:** Compute per-participant per-task eye-tracking features from the pre-aggregated Metrics file and the raw gaze stream.

**Inputs:**
- `data/processed/metrics_clean.parquet`
- `data/processed/raw_gaze_clean.parquet`
- `data/external/task_correct_aoi_map.json`

**Output:**
- `data/processed/features_per_task.parquet` (one row per participant × task)

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import numpy as np
import pandas as pd
from scipy.spatial.distance import euclidean

from src.config import (
    METRICS_CLEAN_PKL, RAWGAZE_CLEAN_PKL, FEATURES_PER_TASK, AOI_MAP_JSON,
    MetricsCols, ExportCols, METRICS_FEATURE_COLS, TASKS
)

pd.set_option('display.max_columns', 50)

## 1. Load inputs

In [ ]:
metrics = pd.read_parquet(METRICS_CLEAN_PKL)
gaze    = pd.read_parquet(RAWGAZE_CLEAN_PKL)

with open(AOI_MAP_JSON) as f:
    aoi_map = json.load(f)

# Strip _comment key
aoi_map = {k: v for k, v in aoi_map.items() if not k.startswith('_')}

print("AOI map loaded for tasks:", list(aoi_map.keys()))
print("Metrics shape:", metrics.shape)
print("Gaze shape:",    gaze.shape)

## 2. Features from Metrics file

For each (Participant, Media/task), extract separate feature rows for:
- correct AOI
- all distractors (aggregated)
- total (all AOIs combined)

In [ ]:
def extract_metrics_features(metrics_df, aoi_map):
    """Return a DataFrame with one row per (participant, task)."""
    records = []

    for task in TASKS:
        if task not in aoi_map:
            print(f"WARNING: {task} not in aoi_map, skipping")
            continue

        correct_aoi    = aoi_map[task]['correct_aoi']
        distractor_aois = aoi_map[task]['distractor_aois']

        task_df = metrics_df[metrics_df[MetricsCols.MEDIA].str.contains(task, na=False, case=False)]

        for participant, grp in task_df.groupby(MetricsCols.PARTICIPANT):
            row = {'participant_id': participant, 'task': task}

            # --- Correct AOI metrics ---
            correct_rows = grp[grp[MetricsCols.AOI].str.strip() == correct_aoi]
            if len(correct_rows) > 0:
                cr = correct_rows.iloc[0]
                for col in METRICS_FEATURE_COLS:
                    row[f'correct_{col}'] = cr.get(col, np.nan)
            else:
                for col in METRICS_FEATURE_COLS:
                    row[f'correct_{col}'] = np.nan

            # --- Distractor metrics (mean across distractor AOIs) ---
            dist_rows = grp[grp[MetricsCols.AOI].str.strip().isin(distractor_aois)]
            if len(dist_rows) > 0:
                for col in [MetricsCols.TOTAL_FIX_DUR, MetricsCols.NUM_FIXATIONS,
                            MetricsCols.TOTAL_VISIT_DUR, MetricsCols.NUM_VISITS]:
                    row[f'distractor_sum_{col}'] = dist_rows[col].sum()
                    row[f'distractor_mean_{col}'] = dist_rows[col].mean()
            else:
                for col in [MetricsCols.TOTAL_FIX_DUR, MetricsCols.NUM_FIXATIONS,
                            MetricsCols.TOTAL_VISIT_DUR, MetricsCols.NUM_VISITS]:
                    row[f'distractor_sum_{col}']  = np.nan
                    row[f'distractor_mean_{col}'] = np.nan

            # --- AOI transition count (proxy: total visits across all AOIs) ---
            row['total_aoi_visits'] = grp[MetricsCols.NUM_VISITS].sum()

            records.append(row)

    return pd.DataFrame(records)


metrics_features = extract_metrics_features(metrics, aoi_map)
print(f"Metrics features shape: {metrics_features.shape}")
metrics_features.head()

## 3. Features from raw gaze stream

Computed per (participant, task):
- `correct_aoi_dwell_ratio` — fraction of total valid frames on correct AOI
- `distractor_dwell_ratio` — fraction on any distractor AOI
- `scanpath_length` — cumulative Euclidean distance between consecutive fixation points
- `refixation_count` — number of returns to correct AOI after leaving
- `pupil_dilation_change` — pupil diameter second-half minus first-half

In [ ]:
def build_aoi_hit_col(task, aoi):
    """Build the column name pattern used in the Data export: 'AOI hit [task - aoi]'."""
    return f"AOI hit [{task} - {aoi}]"


def compute_scanpath_length(fix_x, fix_y):
    """Cumulative Euclidean distance between consecutive fixation centroids."""
    coords = list(zip(fix_x, fix_y))
    if len(coords) < 2:
        return 0.0
    return sum(euclidean(coords[i], coords[i+1]) for i in range(len(coords)-1))


def count_refixations(aoi_hit_series):
    """Count how many times gaze returns to AOI after leaving (0->1 transitions)."""
    transitions = aoi_hit_series.diff().fillna(0)
    return int((transitions == 1).sum())


def extract_gaze_features(gaze_df, aoi_map):
    records = []

    for task in TASKS:
        if task not in aoi_map:
            continue

        correct_aoi     = aoi_map[task]['correct_aoi']
        distractor_aois = aoi_map[task]['distractor_aois']
        correct_col     = build_aoi_hit_col(task, correct_aoi)
        distractor_cols = [build_aoi_hit_col(task, a) for a in distractor_aois
                           if build_aoi_hit_col(task, a) in gaze_df.columns]

        if correct_col not in gaze_df.columns:
            print(f"WARNING: column '{correct_col}' not found — skipping {task}")
            continue

        task_df = gaze_df[gaze_df[ExportCols.PRESENTED_MEDIA].str.contains(task, na=False, case=False)]

        for participant, grp in task_df.groupby(ExportCols.PARTICIPANT_NAME):
            row = {'participant_id': participant, 'task': task}

            total_frames = len(grp)
            valid_frames = grp[ExportCols.GAZE_X].notna().sum()

            # Dwell ratios
            row['correct_aoi_dwell_ratio']    = grp[correct_col].sum() / max(valid_frames, 1)
            if distractor_cols:
                distractor_hits = grp[distractor_cols].any(axis=1).sum()
                row['distractor_dwell_ratio'] = distractor_hits / max(valid_frames, 1)
            else:
                row['distractor_dwell_ratio'] = np.nan

            # Scanpath length (fixation rows only)
            fix_rows = grp[grp[ExportCols.EYE_MOVEMENT_TYPE] == 'Fixation'].dropna(
                subset=[ExportCols.FIXATION_X, ExportCols.FIXATION_Y]
            )
            row['scanpath_length'] = compute_scanpath_length(
                fix_rows[ExportCols.FIXATION_X].values,
                fix_rows[ExportCols.FIXATION_Y].values
            )

            # Re-fixation count on correct AOI
            row['refixation_count'] = count_refixations(grp[correct_col])

            # Pupil dilation change (first half vs second half)
            pupil = grp[ExportCols.PUPIL_FILTERED].dropna()
            if len(pupil) >= 4:
                mid = len(pupil) // 2
                row['pupil_dilation_change'] = pupil.iloc[mid:].mean() - pupil.iloc[:mid].mean()
            else:
                row['pupil_dilation_change'] = np.nan

            row['valid_frames']  = int(valid_frames)
            row['total_frames']  = int(total_frames)

            records.append(row)

    return pd.DataFrame(records)


gaze_features = extract_gaze_features(gaze, aoi_map)
print(f"Gaze features shape: {gaze_features.shape}")
gaze_features.head()

## 4. Merge Metrics and Gaze features

In [ ]:
features = pd.merge(
    metrics_features,
    gaze_features,
    on=['participant_id', 'task'],
    how='outer'
)

print(f"Combined features shape: {features.shape}")
print(f"Participants: {features['participant_id'].nunique()}")
print(f"Tasks: {features['task'].unique()}")
features.head()

## 5. Feature summary

In [ ]:
print("Feature columns:")
feature_cols = [c for c in features.columns if c not in ['participant_id', 'task']]
print(f"  Total: {len(feature_cols)}")
for c in feature_cols:
    print(f"  {c}")

In [ ]:
null_summary = features[feature_cols].isnull().mean().sort_values(ascending=False)
print("\nNull rate per feature:")
null_summary[null_summary > 0]

## 6. Save

In [ ]:
features.to_parquet(FEATURES_PER_TASK, index=False)
print(f"Saved: {FEATURES_PER_TASK}")
print(f"Shape: {features.shape}")